In [ ]:
import os
os.chdir(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))

# Unified Memory


> **Overview**
>
> 
>  * **Time:** 45 min
> 
>  1. Learn about Unified Memory in CUDA.
>  1. Understand how Unified Memory simplifies memory management in CUDA applications.

Unified Memory is a memory management feature in CUDA (CUDA 6 and above) that allows developers to write applications without
worrying about the complexities of managing memory between the host (CPU) and device (GPU).
It provides a single address space for both the host and device, enabling seamless data sharing and access.

In Unified Memory we use `cudaMallocManaged` to allocate memory that is accessible from both the host and device.

```cpp
int *data;
cudaMallocManaged(&data, size * sizeof(int));

// Use data on the host
for (int i = 0; i < size; i++) {
    data[i] = i;
}

// Use data on the device
kernel<<<blocks, threads>>>(data);

// Synchronize to ensure all operations are complete
cudaDeviceSynchronize();
```

Under the hood, Unified Memory automatically migrates data between the host and device as needed.
This means that when the host accesses data that is currently on the device, Unified Memory will
automatically transfer it to the host memory, and vice versa. This migration is managed by the CUDA runtime,
which tracks memory accesses and performs the necessary transfers transparently.

The advantage of Unified Memory is:
* Simplifies programming — no need to manage explicit memory transfers
* Helps porting code from CPU to GPU
* Reduces the complexity of memory management

However, there are some considerations to keep in mind when using Unified Memory:

* May cause performance overhead due to page migration
* Less fine-grained control over memory movement
* Not all CUDA features are compatible with Unified Memory
* Limited support for certain data structures and algorithms

## Managed Memory and Pinned Host Memory


Managed memory is allocated with `cudaMallocManaged`. The runtime manages its accessibility and, where needed, migration between CPU and GPU. `cudaMemAdvise` provides hints for existing memory; it does not allocate memory.

Pinned host memory, allocated with `cudaMallocHost`, is a separate allocation type. It stays in host memory and is page-locked. The example below uses a separate device buffer and explicit copies; it does not rely on the GPU directly accessing a host pointer.

```cpp
// Example using pinned host memory and a separate device buffer
__global__ void kernel(int *data) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    data[idx] += 1; // Increment each element by 1
}

int main() {
    int size = 1024;
    int *data;

    // Allocate page-locked host memory and a separate GPU buffer
    cudaMallocHost(&data, size * sizeof(int));
    int *deviceData;
    cudaMalloc(&deviceData, size * sizeof(int));

    // Initialize data on the host
    for (int i = 0; i < size; i++) {
        data[i] = i;
    }

    cudaMemcpy(deviceData, data, size * sizeof(int), cudaMemcpyHostToDevice);

    // Launch on device memory; size is exactly divisible by 256 here
    kernel<<<(size + 255) / 256, 256>>>(deviceData);

    // Synchronize to ensure all operations are complete
    cudaDeviceSynchronize();

    cudaMemcpy(data, deviceData, size * sizeof(int), cudaMemcpyDeviceToHost);
    cudaFree(deviceData);
    cudaFreeHost(data);
}
```

There are some disadvantages to using pinned memory:

* Pinned memory is not pageable, which means it cannot be swapped out to disk by the operating system.
* It can consume more system memory, as it is not eligible for paging.
* It may lead to reduced system performance if too much pinned memory is used, as it can limit the amount of memory available for other processes.

## cudaMemPrefetchAsync


The `cudaMemPrefetchAsync` function is used to prefetch data from the device to the host or vice versa in a
non-blocking manner. This can help improve performance by ensuring that data is available in the desired
memory space before it is accessed.

```cpp
// Example of using cudaMemPrefetchAsync
int *data;
cudaMallocManaged(&data, size * sizeof(int));

// Prefetch data to the host from GPU 0
cudaMemPrefetchAsync(data, size * sizeof(int), cudaCpuDeviceId);
cudaDeviceSynchronize();  // Wait for the prefetch before CPU access

// Use data on the host
for (int i = 0; i < size; i++) {
    data[i] += 1;
}

// Prefetch data back to GPU 0
cudaMemPrefetchAsync(data, size * sizeof(int), 0);

// Launch kernel on the device
kernel<<<blocks, threads>>>(data);

// Synchronize to ensure all operations are complete
cudaDeviceSynchronize();
```

> **Explanation**
>
> 
>  `cudaCpuDeviceId` is a special device ID that refers to the host CPU.

## cudaMemAdvise


The `cudaMemAdvise` function is used to provide advice to the CUDA runtime about how memory should be managed.

```cpp
// Example of using cudaMemAdvise
int *data;
cudaMallocManaged(&data, N * sizeof(int));

// Initialize data on host
for (int i = 0; i < N; ++i)
    data[i] = i;

// 1. Advise that the data is mostly read, with occasional writes
cudaMemAdvise(data, N * sizeof(int), cudaMemAdviseSetReadMostly, cudaCpuDeviceId);

// 2. Prefer memory to be located on GPU 0
cudaMemAdvise(data, N * sizeof(int), cudaMemAdviseSetPreferredLocation, 0);

// 3. Specify that GPU 0 will access this memory
cudaMemAdvise(data, N * sizeof(int), cudaMemAdviseSetAccessedBy, 0);
```

The different advices that can be provided using `cudaMemAdvise` include:

* `cudaMemAdviseSetReadMostly`: Indicates mostly reads with occasional writes; it is not a CPU-only hint.
* `cudaMemAdviseSetPreferredLocation`: Specifies the preferred location for the memory (host or device).
* `cudaMemAdviseSetAccessedBy`: Indicates which device(s) will access the memory.

```cpp
// Example of using cudaMemAdvise with different advices
int *data;
cudaMallocManaged(&data, size * sizeof(int));

// Advise that the data is mostly read, with occasional writes
cudaMemAdvise(data, size * sizeof(int), cudaMemAdviseSetReadMostly, 0);

// Advise the CUDA runtime that the data will be accessed by device 0
cudaMemAdvise(data, size * sizeof(int), cudaMemAdviseSetAccessedBy, 0);

// Use data on the host
for (int i = 0; i < size; i++) {
    data[i] += 1;
}

// Launch kernel on the device
kernel<<<blocks, threads>>>(data);

// Synchronize to ensure all operations are complete
cudaDeviceSynchronize();
```

> **Key Points**
>
> 
>  1. Unified Memory provides a single address space for both host and device memory.
>  1. It simplifies memory management by automatically migrating data between host and device.
>  1. Use `cudaMallocManaged` to allocate Unified Memory.
>  1. Be aware of potential performance overhead due to automatic page migration.
>  1. Not all CUDA features are compatible with Unified Memory, so check compatibility when using it.
>  1. Pinned memory can improve performance for certain operations but requires careful management.
>  1. Use `cudaMemPrefetchAsync` to prefetch data between host and device in a non-blocking manner.
>  1. Use `cudaMemAdvise` to provide advice to the CUDA runtime about memory management.

## Notes on the examples

The notebook clarifies the source's distinction between managed allocations and pinned host allocations, corrects the role of `cudaMemAdvise`, and waits for host prefetch completion before CPU access. See [NVIDIA CUDA 12.9 memory management](https://docs.nvidia.com/cuda/archive/12.9.0/cuda-runtime-api/group__CUDART__MEMORY.html).

The tutorial snippets illustrate API usage and omit some setup, error handling, or cleanup. The exercise below uses a complete program. Its prefetch calls follow the CUDA 12.9 API used by this workshop.

## Exercise 08: Unified Memory
**Time: 20 min**

1. Run [exercises/08-unified-memory/vector_add_unified.cu](../exercises/08-unified-memory/vector_add_unified.cu) and check the output.
2. Do you see any difference in performance when prefetching data to the GPU?

Use a notebook kernel on an allocated GPU node with CUDA loaded and `nvcc` available. The supplied program uses managed allocations, four streams, and ten iterations. Its prefetch calls are commented out initially.

### Compile and run the baseline

Check that result verification reports `PASSED` and record the total time.

In [ ]:
from pathlib import Path

output_dir = Path("unified_memory_output")
output_dir.mkdir(exist_ok=True)
!nvcc -O2 exercises/08-unified-memory/vector_add_unified.cu -o unified_memory_output/baseline

In [ ]:
!./unified_memory_output/baseline

### Enable GPU prefetching in a separate copy

The next cell enables the three prefetch calls for `A`, `B`, and `C` to GPU 0, keeping the exercise source unchanged. The optional prefetch back to the CPU remains disabled in both versions, so it does not change this comparison.

In [ ]:
import re

source_path = Path("exercises/08-unified-memory/vector_add_unified.cu")
source = source_path.read_text()
prefetch_source, enabled = re.subn(
    r"(?m)^(\s*)//(cudaMemPrefetchAsync\([ABC]\[iter\], size, 0, streams\[stream_id\]\);)$",
    r"\1\2", source,
)
assert enabled == 3, f"Expected three disabled GPU-prefetch calls, found {enabled}"
(output_dir / "with_prefetch.cu").write_text(prefetch_source)
print("Enabled GPU prefetching for A, B, and C.")

In [ ]:
!nvcc -O2 unified_memory_output/with_prefetch.cu -o unified_memory_output/with_prefetch

In [ ]:
!./unified_memory_output/with_prefetch

### Compare results

Rerun each executable a few times on the same allocated GPU. Verify that both versions report `PASSED`, then record their printed times.

| Run | No explicit prefetch (ms) | GPU prefetch enabled (ms) |
| --- | --- | --- |
| 1 | | |
| 2 | | |
| 3 | | |

1. Is prefetching consistently faster for this input size?
2. Why are the prefetch calls submitted to the same stream as the kernel that uses the data?
3. What changes when you also enable the optional result prefetch back to the CPU?

The reported overall event interval covers multiple streams and host submission/initialization gaps; it is not a kernel-only measurement. CPU result verification happens after the timer stops. Enabling a return-to-CPU prefetch moves that migration into the measured interval, so keep that setting explicit when comparing results.

### Observations

Record timings, correctness, and whether prefetching changed the measured performance.